In [1]:
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"
file_name = "yellow_tripdata_2024-10.parquet"

# Download the file
urllib.request.urlretrieve(url, file_name)

('yellow_tripdata_2024-10.parquet', <http.client.HTTPMessage at 0x216e6aef8e0>)

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')

df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-09-30 20:30:44|  2024-09-30 20:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [3]:
spark.version

'3.3.2'

In [4]:
df_repartitioned = df.repartition(4)
df_repartitioned.write.mode("overwrite").parquet("repartitioned_yellow_tripdata_2024-10.parquet")


In [15]:
from pyspark.sql.functions import to_date
q3df = df.filter(to_date(df.tpep_pickup_datetime) == '2024-10-15').dropDuplicates()
q3df.count()

128909

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import max
df=df.withColumn("trip_hour", (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 3600)
df.select(max("trip_hour")).collect()[0][0]

[Row(max(trip_hour)=162.61777777777777)]

In [26]:
# http://localhost:4040/jobs/

In [27]:
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
file_name = "taxi_zone_lookup.csv"
urllib.request.urlretrieve(url, file_name)

('taxi_zone_lookup.csv', <http.client.HTTPMessage at 0x21696c8ffa0>)

In [30]:
taxi_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

taxi_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [50]:
df_taxi_zone = df.join(taxi_zone, df.PULocationID == taxi_zone.LocationID, "left")

In [49]:
result = (df_taxi_zone
          .groupBy("Zone")
          .count()
          .orderBy(col("count"))
          .limit(3))
result.show()



+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Rikers Island|    2|
|       Arden Heights|    2|
+--------------------+-----+

